In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass

plt.rcParams["figure.figsize"] = (8, 6)
plt.rcParams["axes.grid"] = True


In [ ]:
def load_macro_from_npz(npz_path, axes=(0, 1)):
    data = np.load(npz_path, allow_pickle=False)

    Lambda = data["Lambda_xyz"]  # (3,3)
    Dx     = data["Dx_xyz"]      # (3,3)
    Kx     = data["Kx_xyz"]      # (3,3)

    a0, a1 = axes

    M2 = Lambda[np.ix_([a0, a1], [a0, a1])]
    C2 = Dx[np.ix_([a0, a1], [a0, a1])]
    K2 = Kx[np.ix_([a0, a1], [a0, a1])]

    extras = {k: data[k] for k in data.files if k not in ["Lambda_xyz", "Dx_xyz", "Kx_xyz"]}
    return M2, C2, K2, extras


def rotation_theta_z(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s],
                     [s,  c]])


def state_matrix(M, C, K_eff):
    zeros = np.zeros((2, 2))
    I = np.eye(2)
    Minv = np.linalg.inv(M)
    A = np.block([
        [zeros,             I],
        [-Minv @ K_eff, -Minv @ C]
    ])
    return A


def compute_entry_exit_angles_downmilling(r_engagement: float, v: float, D: float):
    # Mirrors your original function: forces full-entry by setting v=0 internally.
    R = D / 2
    b = r_engagement
    v = 0.0

    arg  = np.clip((R - b) / R, -1.0, 1.0)
    arg1 = np.clip(v / (D/2), 0.0, 1.0)

    phi_st = np.pi / 2 + np.arcsin(arg)
    phi_ex = np.pi / 2 + np.arccos(arg1)

    return phi_st, phi_ex


def zero_order_dF(
    c: float,
    a_axial: float,
    b_radial: float,
    D: float,
    N_teeth: float,
    phi_st: float,
    phi_ex: float,
    n_hat: np.ndarray,
    Ktc: float,
    Krc: float
):
    nx, ny = n_hat
    tx, ty = -ny, nx
    v = 0.0  # full entry assumption

    dphi_st_db = -1.0 / np.sqrt(b_radial * (D - b_radial))
    dphi_ex_dv = -2.0 / np.sqrt(D**2 - 4*v**2)

    def q(phi):
        h = c * np.sin(phi)
        T = np.array([[-np.cos(phi), -np.sin(phi)],
                      [ np.sin(phi), -np.cos(phi)]])
        K = np.array([Ktc, Krc])
        return a_axial * (T @ K) * h

    dF_dv = N_teeth/(2*np.pi) * q(phi_ex) * dphi_ex_dv
    dF_db = N_teeth/(2*np.pi) * (-q(phi_st) * dphi_st_db)

    dF_dx = dF_db * (-nx) + dF_dv * (-tx)
    dF_dy = dF_db * (-ny) + dF_dv * (-ty)

    return np.column_stack((dF_dx, dF_dy))


def max_real_eig(M, C, K, Kp0, theta):
    R = rotation_theta_z(theta)
    Kp = R @ Kp0 @ R.T
    K_eff = K - Kp
    A = state_matrix(M, C, K_eff)
    eigvals = np.linalg.eigvals(A)
    return np.max(np.real(eigvals))


def eigenvalues_for_a(M2, C2, K2, theta, a_vals,
                      c_feed, b_radial, D, N_teeth,
                      phi_st, phi_ex, Ktc, Krc, n_hat,
                      scale_Kp=1000.0):

    eigvals_all = np.zeros((len(a_vals), 4), dtype=complex)
    max_real = np.zeros(len(a_vals), dtype=float)

    for i, aax in enumerate(a_vals):
        Kp0 = zero_order_dF(
            c=c_feed, a_axial=aax, b_radial=b_radial, D=D, N_teeth=N_teeth,
            phi_st=phi_st, phi_ex=phi_ex, Ktc=Ktc, Krc=Krc, n_hat=n_hat
        )
        Kp0 = scale_Kp * Kp0

        R = rotation_theta_z(theta)
        Kp = R @ Kp0 @ R.T
        K_eff = K2 - Kp
        A = state_matrix(M2, C2, K_eff)

        eigvals = np.linalg.eigvals(A)

        idx = np.lexsort((np.real(eigvals), np.imag(eigvals)))
        eigvals = eigvals[idx]

        eigvals_all[i, :] = eigvals
        max_real[i] = np.max(np.real(eigvals))

    return eigvals_all, max_real


In [ ]:
NPZ_PATH = "./data/linearized_operational_space_xyz.npz"
AXES = (0, 1)   # (x,y)

Ktc = 1930.4
Krc = 1159.6

b_radial = 10.0   # mm
D = 20.0          # mm
N_teeth = 8

feed_mm_s = 10.0  # mm/s
rpm = 400.0       # rpm

theta_step_deg = 1.0

a_max = 100.0     # mm
delta_a = 1.0     # mm

SCALE_KP = 1000.0

n_hat = np.array([0.0, 1.0])  # edge surface normal in XY


In [ ]:
theta_deg = np.arange(0.0, 360.0 + theta_step_deg, theta_step_deg)
thetas = np.deg2rad(theta_deg)

n_a = int(a_max / delta_a) + 1
a_vals = np.linspace(0.0, a_max, n_a)

TH, AAX = np.meshgrid(thetas, a_vals, indexing="xy")
TH_flat = TH.ravel()
A_flat  = AAX.ravel()

stable_mask = np.empty_like(A_flat, dtype=bool)

for i, (th, aax) in enumerate(zip(TH_flat, A_flat)):
    Kp0 = zero_order_dF(
        c=c_feed, a_axial=aax, b_radial=b_radial, D=D, N_teeth=N_teeth,
        phi_st=phi_st, phi_ex=phi_ex, Ktc=Ktc, Krc=Krc, n_hat=n_hat
    )
    Kp0 = SCALE_KP * Kp0
    stable_mask[i] = (max_real_eig(M2, C2, K2, Kp0, th) < 0.0)

print(f"Stable points: {stable_mask.sum()}/{stable_mask.size} ({stable_mask.mean()*100:.1f}%)")


NameError: name 'c_feed' is not defined

In [ ]:
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="polar")

ax.scatter(TH_flat[stable_mask],  A_flat[stable_mask],  s=10, marker="o", alpha=0.6, label="Stable")
ax.scatter(TH_flat[~stable_mask], A_flat[~stable_mask], s=14, marker="x", alpha=0.9, label="Unstable")

ax.set_title("Stability map vs orientation (angle) and axial depth (radius)")
ax.set_rlabel_position(22.5)
ax.set_theta_zero_location("E")
ax.set_theta_direction(1)
ax.set_rlim(0, a_max)
ax.legend(loc="upper right", bbox_to_anchor=(1.25, 1.1))

plt.show()


In [ ]:
theta_inspect_deg = 0.0
theta_inspect = np.deg2rad(theta_inspect_deg)

eigvals_all, max_real = eigenvalues_for_a(
    M2, C2, K2, theta_inspect, a_vals,
    c_feed=c_feed, b_radial=b_radial, D=D, N_teeth=N_teeth,
    phi_st=phi_st, phi_ex=phi_ex, Ktc=Ktc, Krc=Krc, n_hat=n_hat,
    scale_Kp=SCALE_KP
)

plt.figure()
plt.plot(a_vals, max_real)
plt.axhline(0.0, linestyle="--")
plt.xlabel("Axial depth a [mm]")
plt.ylabel("max Re(eig(A))")
plt.title(f"Stability metric vs axial depth (theta = {theta_inspect_deg:.1f} deg)")
plt.grid(True)
plt.show()


NameError: name 'M2' is not defined

In [ ]:
plt.figure()
for k in range(eigvals_all.shape[1]):
    plt.plot(np.real(eigvals_all[:, k]), np.imag(eigvals_all[:, k]),
             marker=".", markersize=3, linestyle="-")
plt.axvline(0.0, linestyle="--")
plt.xlabel("Re(lambda)")
plt.ylabel("Im(lambda)")
plt.title(f"Eigenvalue trajectories as a increases (theta = {theta_inspect_deg:.1f} deg)")
plt.grid(True)
plt.show()


NameError: name 'eigvals_all' is not defined

<Figure size 800x600 with 0 Axes>

In [ ]:
plt.figure()
for k in range(eigvals_all.shape[1]):
    plt.plot(a_vals, np.real(eigvals_all[:, k]), marker=".", markersize=3, linestyle="-")
plt.axhline(0.0, linestyle="--")
plt.xlabel("Axial depth a [mm]")
plt.ylabel("Re(lambda)")
plt.title(f"Real parts of eigenvalues vs a (theta = {theta_inspect_deg:.1f} deg)")
plt.grid(True)
plt.show()


NameError: name 'eigvals_all' is not defined

<Figure size 800x600 with 0 Axes>